#Ingest races.csv file
1. Read the file using spark dataframe reader API
2. Add Metadata columns:
    - Source File
    - Ingestion Timestamp
3. Write to bronze delta table


In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/races.csv"
table_name = f"{catalog_name}.{bronze_schema}.races"

#### Step 1 -  Read the CSV file using the dataframe reader API

In [0]:
from pyspark.sql.types import *
races_schema = StructType([
    StructField('season', IntegerType(), True),
    StructField('round', IntegerType(), True),
    StructField('url', StringType(), True),
    StructField('raceName', StringType(), True),
    StructField('date', DateType(), True),
    StructField('circuitId', StringType(), True)
])



In [0]:
races_df= (
    spark.read
    .format('csv')
    .option('header','true')
    #.option('inferSchema','true')
    .option('mode','FAILFAST')
    .schema(races_schema)
    .load(source_file))

In [0]:
display(races_df)

#### Step 2 - Add Metadata columns:
  - Source File
  - Ingestion Timestamp

In [0]:
races_final_df = add_ingestion_metadata(races_df)

In [0]:
display(races_final_df)

#### Step 3 -  Write to bronze delta table

In [0]:
write_to_bronze(races_final_df, table_name, v_batch_id)

In [0]:
display(spark.read.table(table_name))